In [ ]:
import data_poisoning_attack

c:\Users\uiv09218\AppData\Local\miniforge3\envs\xai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
Training model...
Model test accuracy: 0.7273
Preparing attack prerequisites...
Starting evolution...
Start population fitness calculation:
	Concatenating population took 0.0035686492919921875 seconds.
	Producing SHAP explanations took 365.92802929878235 seconds.
	Producing SHAP reference explanations took 0.0 seconds.
	Calculating drift scores took 0.03689384460449219 seconds.
	Calculating estimated probs took 0.0 seconds.
	Calculating LCB took 0.0 seconds.
Calculating optimization metrics took 365.96849179267883 seconds.
Initial Population - Best Fitness: 2.719547 --- Time: 365.97s
before evolution --- Time: 365.97s
--- produced next generation --- Time: 366.02s
Start population fitness calculation:
	Concatenating population took 0.0 seconds.
	Producing SHAP explanations took 369.4089984893799 seconds.
	Producing SHAP reference explanations took 0.0 seconds.
	Calculating drift scores took 0.031252384185791016 seconds.
	Calculating estimated probs took 0.0 seconds.
	Ca

TypeError: Individual.mutate() got an unexpected keyword argument 'cat_mask'

In [51]:
import pickle
import data_poisoning as dp
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import copy
import numpy as np

In [52]:
with open("evolution_results_25p_4s_40g_0.3dt_0.95dc.pkl", "rb") as f:
    eval_results = pickle.load(f)

In [53]:
eval_results['early_stopping'].best_stds

array([0.017258  , 0.01843743, 0.02351082, 0.01751147, 0.01830297,
       0.02947557, 0.02756795, 0.02494899])

In [54]:
eval_individual = dp.Individual(
    data=eval_results['X_train'].values.copy(),
    mutation_rate=1.0,
    mutation_stds=eval_results['early_stopping'].best_stds
)

sample = [copy.deepcopy(eval_individual) for _ in range(50)]

for s in sample:
    s.mutate(
        X_min=eval_results['X_train'].values.min(axis=0),
        X_max=eval_results['X_train'].values.max(axis=0),
        X_cat=eval_results['X_train'].shape[1]*[None]
    )

In [55]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=3368494378,
    n_jobs=-1
)

In [56]:
model.fit(eval_results['X_train'].values, eval_results['y_train'].values)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [58]:
explainer = dp.shap.Explainer(model)

reference_explanations = dp.produce_shap_explanations(
    explainer,
    eval_results['X_train'].values
)

In [62]:
mutation_explanations = [dp.produce_shap_explanations(
        explainer,
        s.data
    ).mean(axis=0) for s in sample
]

In [76]:
total_mutation_explanations = np.expand_dims(np.concatenate(np.expand_dims(mutation_explanations, axis=0), axis=0), axis=0)
reference_explanations_rep = np.tile(
    reference_explanations.mean(axis=0),
    (1, 50, 1)
)

In [77]:
total_mutation_explanations.shape, reference_explanations_rep.shape

((1, 50, 8), (1, 50, 8))

In [78]:
drift_scores = dp.explanation_drift(
    total_mutation_explanations,
    reference_explanations_rep
)

In [84]:
drift_scores

array([[0.23324284, 0.05391513, 0.26863757, 0.16169471, 0.0719909 ,
        0.17936455, 0.05427019, 0.1616427 , 0.17931652, 0.05408337,
        0.072088  , 0.16127696, 0.05405444, 0.12570975, 0.16135777,
        0.16145185, 0.19701595, 0.05410217, 0.05416579, 0.0541056 ,
        0.16142563, 0.21502965, 0.05409188, 0.05398631, 0.14327862,
        0.07180695, 0.10822702, 0.17950211, 0.05408944, 0.14338426,
        0.16145741, 0.05400858, 0.26869387, 0.07201866, 0.16164867,
        0.05402154, 0.17915889, 0.07198954, 0.16136185, 0.17923847,
        0.17912014, 0.1790885 , 0.16131307, 0.07184842, 0.16143145,
        0.05420418, 0.2151716 , 0.05407217, 0.26854387, 0.17934506]])

In [95]:
estimated_probabilities = dp.estimate_probability(
    drift_scores = drift_scores,
    threshold = 0.1
)

In [96]:
estimated_probabilities

array([0.6])

In [97]:
lcb = dp.LCB_Wilson(
    estimated_probability = estimated_probabilities,
    sample_size = 50
)

In [98]:
lcb

array([0.33601027])

In [57]:
reference_predictions = model.predict(eval_results['X_train'].values)
mutation_predictions = model.predict(eval_individual.data)
correct_prediction = reference_predictions == mutation_predictions

print(correct_prediction.mean())

1.0


In [ ]:
global_mutation_explanations = dp.produce_shap_explanations(
    explainer,
    eval_individual.data
)[correct_prediction]

print("r:", reference_explanations.mean(axis=0).shape)
print("m:", global_mutation_explanations.mean(axis=0).shape)

r: (8,)
m: (8,)


In [ ]:
dp.explanation_drift(
    mutation_explanations=global_mutation_explanations,
    reference_explanations=reference_explanations,
)

In [ ]:
eval_plot = dp.plot_evaluation(
    global_reference_explanations=reference_explanations.mean(axis=0),
    global_mutation_explanations=global_mutation_explanations.mean(axis=0),
    standard_deviations=eval_individual.mutation_stds,
    feature_names=eval_results['X_train'].columns.tolist(),
    drift_threshold=0.3,
    drift_confidence=0.95
)